In [ ]:
!pip cache purge

Files removed: 0


In [ ]:
!pip uninstall -y langgraph langgraph-prebuilt langchain-classic
!pip install -qU langchain langchain-core langchain-community langchain-groq langchain-huggingface langchain-chroma chromadb transformers sentence-transformers pypdf python-docx rank-bm25 tqdm pandas==2.2.2 requests==2.32.4

Found existing installation: langgraph 1.1.10
Uninstalling langgraph-1.1.10:
  Successfully uninstalled langgraph-1.1.10
Found existing installation: langgraph-prebuilt 1.0.12
Uninstalling langgraph-prebuilt-1.0.12:
  Successfully uninstalled langgraph-prebuilt-1.0.12
Found existing installation: langchain-classic 1.0.4
Uninstalling langchain-classic-1.0.4:
  Successfully uninstalled langchain-classic-1.0.4


In [ ]:
!pip install -qU langchain langchain-core langchain-community langchain-groq langchain-huggingface langchain-chroma chromadb transformers sentence-transformers pypdf python-docx rank-bm25 tqdm pandas==2.2.2 requests==2.32.4

In [ ]:
# ==========================================================
# CELL 2 - IMPORTS
# ==========================================================
import os
import re
import json
import glob
import time
import numpy as np
import pandas as pd
from tqdm import tqdm


from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from rank_bm25 import BM25Okapi
from pypdf import PdfReader
from docx import Document as DocxDocument

# ==========================================================
# CELL 3 - SET API KEY
# ==========================================================
os.environ["GROQ_API_KEY"] = "key"


In [ ]:
# ==========================================================
# CELL 4 - LOAD GROQ MODEL
# ==========================================================
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [ ]:
RESUME_FOLDER = "/content"
DB_FOLDER = "/content/chroma_db"

os.makedirs(DB_FOLDER, exist_ok=True)

In [ ]:

# ==========================================================
# CELL 6 - FILE READERS
# ==========================================================
def read_pdf(path):
    text = ""
    reader = PdfReader(path)
    for page in reader.pages:
        t = page.extract_text()
        if t:
            text += t + "\n"
    return text

def read_docx(path):
    doc = DocxDocument(path)
    return "\n".join([p.text for p in doc.paragraphs])

def read_txt(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def load_file(path):
    p = path.lower()

    if p.endswith(".pdf"):
        return read_pdf(path)

    elif p.endswith(".docx"):
        return read_docx(path)

    elif p.endswith(".txt"):
        return read_txt(path)

    return ""


In [ ]:
# ==========================================================
# CELL 7 - RECURSIVE CHUNKING
# ==========================================================
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=250,
    separators=[
        "\nEducation",
        "\nExperience",
        "\nSkills",
        "\nProjects",
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

def chunk_text(text):
    return text_splitter.split_text(text)

In [ ]:
# ==========================================================
# CELL 8 - LOAD DOCUMENTS (SAFE)
# ==========================================================
docs = []

files = glob.glob("/content/*")
print("Files found:", files)

for file in files:
    lower = file.lower()

    if lower.endswith(".pdf") or lower.endswith(".docx") or lower.endswith(".txt"):

        print("Reading:", file)
        text = load_file(file)

        if text is None:
            continue

        text = text.strip()

        if len(text) == 0:
            print("Skipped empty file:", file)
            continue

        chunks = chunk_text(text)

        for chunk in chunks:
            chunk = chunk.strip()

            if len(chunk) > 20:   # minimum useful chunk
                docs.append(
                    Document(
                        page_content=chunk,
                        metadata={"source": file}
                    )
                )

print("Total valid docs:", len(docs))

Files found: ['/content/resume_23_Rohan_Joshi.pdf', '/content/resume_6_Sarah_Kumar.pdf', '/content/resume_10_John_Gupta.pdf', '/content/resume_29_Kavya_Rao.pdf', '/content/resume_16_Aarav_Brown.pdf', '/content/resume_1_David_Johnson.pdf', '/content/resume_21_Neha_Joshi.pdf', '/content/resume_9_Aditya_Sharma.pdf', '/content/resume_8_Priya_Sharma.pdf', '/content/resume_19_John_Desai.pdf', '/content/resume_5_Vikram_Patel.pdf', '/content/resume_27_Vikram_Desai.pdf', '/content/resume_12_Rahul_Patel.pdf', '/content/resume_22_Sneha_Smith.pdf', '/content/resume_15_Vikram_Sharma.pdf', '/content/resume_17_David_Patel.pdf', '/content/chroma_db', '/content/resume_2_Emma_Sharma.pdf', '/content/resume_25_Neha_Rao.pdf', '/content/resume_20_Vikram_Johnson.pdf', '/content/resume_4_Neha_Johnson.pdf', '/content/resume_7_Aarav_Joshi.pdf', '/content/resume_11_David_Patel.pdf', '/content/resume_3_Rohan_Reddy.pdf', '/content/resume_13_Rohan_Sharma.pdf', '/content/resume_26_Neha_Brown.pdf', '/content/resume_3

In [ ]:
# ==========================================================
# CELL 9 - EMBEDDINGS
# ==========================================================
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [ ]:
# ==========================================================
# CELL 10 - CHROMA DB
# ==========================================================
if len(docs) == 0:
    raise ValueError("No valid documents found. Upload TXT/DOCX/PDF resumes.")

vectordb = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    persist_directory="/content/chroma_db"
)

print("Chroma DB Ready")

Chroma DB Ready


In [ ]:
# ==========================================================
# CELL 11 - BM25 INDEX
# ==========================================================
texts = [d.page_content for d in docs]
tokenized = [t.lower().split() for t in texts]

bm25 = BM25Okapi(tokenized)

In [ ]:
# ==========================================================
# CELL 12 - HYBRID RETRIEVAL
# ==========================================================
def retrieve_candidates(job_description, top_k=10):

    retriever = vectordb.as_retriever(
        search_type="mmr",
        search_kwargs={"k": top_k}
    )

    semantic_docs = retriever.invoke(job_description)

    keyword_scores = bm25.get_scores(job_description.lower().split())
    keyword_idx = np.argsort(keyword_scores)[::-1][:top_k]

    keyword_docs = [docs[i] for i in keyword_idx]

    merged = semantic_docs + keyword_docs

    unique = {}

    for d in merged:
        unique[d.metadata["source"]] = d

    return list(unique.values())[:top_k]


In [ ]:
# ==========================================================
# CELL 13 - JOB MATCHER
# ==========================================================
def match_candidates(job_description, top_k=10):

    start = time.time()

    retrieved = retrieve_candidates(job_description, top_k)

    context = ""

    for i, doc in enumerate(retrieved, 1):

        context += f"""
Candidate {i}
Resume Path: {doc.metadata["source"]}

Resume Content:
{doc.page_content}

-------------------------
"""

    prompt = f"""
You are an expert recruiter and ATS system.

Analyze all candidates against the job description.

JOB DESCRIPTION:
{job_description}

CANDIDATES:
{context}

Return ONLY valid JSON:

{{
  "job_description": "...",
  "top_matches": [
    {{
      "candidate_name": "...",
      "resume_path": "...",
      "match_score": 0,
      "matched_skills": ["..."],
      "relevant_excerpts": ["..."],
      "reasoning": "..."
    }}
  ]
}}

Rules:
- Rank highest first
- Score 0 to 100
- Infer candidate names from resume text
- Use factual evidence
- Return max {top_k} candidates
"""

    response = llm.invoke(prompt)
    output = response.content.strip()

    latency = round(time.time() - start, 2)

    try:
        result = json.loads(output)
        result["latency_seconds"] = latency
        return result
    except:
        return {
            "raw_output": output,
            "latency_seconds": latency
        }


In [ ]:
# ==========================================================
# CELL 14 - TEST
# ==========================================================
jd = """
Need Python Machine Learning Engineer with 5+ years experience.
Must know SQL, Docker, AWS, REST APIs.
"""

result = match_candidates(jd, top_k=10)
print(result)
print(json.dumps(result, indent=2))



{'raw_output': '```json\n{\n  "job_description": "Need Python Machine Learning Engineer with 5+ years experience. Must know SQL, Docker, AWS, REST APIs.",\n  "top_matches": [\n    {\n      "candidate_name": "David Johnson",\n      "resume_path": "/content/resume_1_David_Johnson.pdf",\n      "match_score": 90,\n      "matched_skills": ["Python", "SQL", "Docker"],\n      "relevant_excerpts": ["Developed and maintained critical systems using SQL and PyTorch.", "7 years of experience specializing in Machine Learning Engineer"],\n      "reasoning": "David Johnson has 7 years of experience in Machine Learning Engineering and has worked with SQL, Docker, and PyTorch, which are relevant to the job description."\n    },\n    {\n      "candidate_name": "Vikram Sharma",\n      "resume_path": "/content/resume_15_Vikram_Sharma.pdf",\n      "match_score": 85,\n      "matched_skills": ["Python", "SQL", "Docker", "NLP"],\n      "relevant_excerpts": ["Developed and maintained critical systems using NLP

In [ ]:
# ==========================================================
# CELL 15 - SAVE JSON
# ==========================================================
with open("/content/matching_results.json", "w") as f:
    json.dump(result, f, indent=2)

print("Saved to /content/matching_results.json")



Saved to /content/matching_results.json


In [ ]:
# ==========================================================
# CELL 16 - INTERACTIVE SEARCH
# ==========================================================
user_jd = input("Paste Job Description: ")

final_result = match_candidates(user_jd, top_k=10)

print(json.dumps(final_result, indent=2))

In [ ]:
# ==========================================================
# CELL 14 - TEST (Clean Output Handling)
# ==========================================================
import json
import re

jd = """
Need Python Machine Learning Engineer with 5+ years experience.
Must know SQL, Docker, AWS, REST APIs.
"""

result = match_candidates(jd, top_k=10)

# If function returns string / markdown json block
raw = result["raw_output"] if isinstance(result, dict) and "raw_output" in result else result

# Remove ```json ... ```
if isinstance(raw, str):
    cleaned = re.sub(r"```json|```", "", raw).strip()
    parsed = json.loads(cleaned)
else:
    parsed = raw

# Pretty print
print("="*80)
print("JOB DESCRIPTION")
print("="*80)
print(parsed["job_description"])

print("\n" + "="*80)
print("TOP MATCHED CANDIDATES")
print("="*80)

for i, c in enumerate(parsed["top_matches"], 1):
    print(f"\n{i}. {c['candidate_name']}  | Score: {c['match_score']}%")
    print(f"Resume: {c['resume_path']}")
    print(f"Skills: {', '.join(c['matched_skills'])}")
    print("Relevant:")
    for ex in c["relevant_excerpts"]:
        print(f"  - {ex}")
    print(f"Reason: {c['reasoning']}")

# Optional full json
print("\n" + "="*80)
print("FULL JSON")
print("="*80)
print(json.dumps(parsed, indent=2))


# ==========================================================
# CELL 15 - SAVE JSON
# ==========================================================
with open("/content/matching_results.json", "w") as f:
    json.dump(parsed, f, indent=2)

print("\nSaved to /content/matching_results.json")

JOB DESCRIPTION
Need Python Machine Learning Engineer with 5+ years experience. Must know SQL, Docker, AWS, REST APIs.

TOP MATCHED CANDIDATES

1. David Johnson  | Score: 90%
Resume: /content/resume_1_David_Johnson.pdf
Skills: Python, SQL, Docker
Relevant:
  - Developed and maintained critical systems using SQL and PyTorch.
  - 7 years of experience specializing in Machine Learning Engineer
Reason: David Johnson has 7 years of experience in Machine Learning Engineering and has worked with SQL, Docker, and PyTorch, which are relevant to the job description.

2. Vikram Sharma  | Score: 85%
Resume: /content/resume_15_Vikram_Sharma.pdf
Skills: Python, SQL, Docker, NLP
Relevant:
  - Developed and maintained critical systems using NLP and Docker.
  - 7 years of experience specializing in Machine Learning Engineer
Reason: Vikram Sharma has 7 years of experience in Machine Learning Engineering and has worked with SQL, Docker, and NLP, which are relevant to the job description.

3. Priya Sharma